In [1]:
!pip install -q transformers datasets sentencepiece evaluate sacrebleu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00


In [3]:
# Load dataset directly from github

from datasets import load_dataset

dataset = load_dataset("freococo/burmese-contextual-pragmatics")
dataset

DatasetDict({
    train: Dataset({
        features: ['uid', 'root_meaning', 'instruction', 'context', 'style', 'tone', 'utterance', 'phonetic', 'register', 'politeness', 'emotion', 'power_relation', 'intent_strength', 'notes'],
        num_rows: 2200
    })
})

In [4]:
train_ds = dataset["train"] if "train" in dataset else dataset[list(dataset.keys())[0]]
train_ds.column_names

['uid',
 'root_meaning',
 'instruction',
 'context',
 'style',
 'tone',
 'utterance',
 'phonetic',
 'register',
 'politeness',
 'emotion',
 'power_relation',
 'intent_strength',
 'notes']

In [5]:
print(train_ds[0])

{'uid': 'bur_beau_001', 'root_meaning': 'You are so beautiful', 'instruction': 'Simple standard compliment.', 'context': 'General social interaction.', 'style': 'Standard', 'tone': 'Neutral', 'utterance': 'မင်း တကယ် လှတယ်။', 'phonetic': 'Min ta-kal hla tal', 'register': 'colloquial', 'politeness': 'neutral', 'emotion': 'admiration', 'power_relation': 'equal', 'intent_strength': 'normal', 'notes': "The most common way to say 'You are really beautiful'."}


In [6]:
# Unique values per attributes

from collections import Counter

columns_to_check = ["politeness", "tone", "power_relation", "context"]

for col in columns_to_check:
    values = [str(x).strip().lower() for x in train_ds[col]]
    counts = Counter(values)

    print(f"\n=== {col} ===")
    print(f"Total unique: {len(counts)}")
    print("Top 10 most common:")
    for k, v in counts.most_common(10):
        print(f"{k}: {v}")


=== politeness ===
Total unique: 13
Top 10 most common:
neutral: 1165
polite: 373
informal: 341
professional: 94
blunt: 70
rude: 53
very_polite: 49
friendly: 33
religious: 11
impolite: 7

=== tone ===
Total unique: 131
Top 10 most common:
neutral: 735
casual: 171
direct: 149
warm: 112
professional: 104
serious: 83
sweet: 72
cheerful: 64
soft: 61
sincere: 47

=== power_relation ===
Total unique: 8
Top 10 most common:
equal: 1643
inferior_to_superior: 207
any: 177
superior_to_inferior: 162
customer_to_staff: 6
customer_to_seller: 3
passenger_to_driver: 1
patient_to_doctor: 1

=== context ===
Total unique: 1660
Top 10 most common:
general.: 21
addressing an elder or superior.: 20
travel.: 19
romantic.: 18
domestic.: 15
general social interaction.: 11
family.: 11
social.: 11
parent to child.: 9
to a friend.: 9


In [10]:
# Build Prompt

def build_prompt(example):
    meaning = str(example["root_meaning"]).strip()
    context = str(example["context"]).strip()
    politeness = str(example["politeness"]).strip().lower()
    tone = str(example["tone"]).strip().lower()
    power = str(example["power_relation"]).strip().lower()
    target = str(example["utterance"]).strip()

    prompt = (
        "Generate Burmese sentence:\n"
        f"Meaning: {meaning}\n"
        f"Context: {context}\n"
        f"Politeness: {politeness}\n"
        f"Tone: {tone}\n"
        f"Power: {power}"
    )

    return {
        "input_text": prompt,
        "target_text": target
    }

In [11]:
processed = train_ds.map(build_prompt)

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [12]:
print("INPUT:\n", processed[0]["input_text"])
print("\nTARGET:\n", processed[0]["target_text"])

INPUT:
 Generate Burmese sentence:
Meaning: You are so beautiful
Context: General social interaction.
Politeness: neutral
Tone: neutral
Power: equal

TARGET:
 မင်း တကယ် လှတယ်။


In [13]:
# TTS

split1 = processed.train_test_split(test_size=0.3, seed=42)
train_data = split1["train"]
temp_data = split1["test"]

split2 = temp_data.train_test_split(test_size=0.5, seed=42)
valid_data = split2["train"]
test_data = split2["test"]

print(len(train_data), len(valid_data), len(test_data))

1540 330 330


In [14]:
# Load tokenizer and model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
print(type(tokenizer).__name__)
print(type(model).__name__)

T5Tokenizer
MT5ForConditionalGeneration


In [20]:
# Tokenize

max_input_length = 128
max_target_length = 64

def tokenize_function(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    # mask padding tokens so loss ignores them
    label_ids = labels["input_ids"]
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        if isinstance(seq, list) else label_ids
        for seq in [label_ids]
    ][0]

    model_inputs["labels"] = label_ids
    return model_inputs

In [21]:
tokenized_train = train_data.map(tokenize_function)
tokenized_valid = valid_data.map(tokenize_function)
tokenized_test  = test_data.map(tokenize_function)

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

In [23]:
keep_cols = ["input_ids", "attention_mask", "labels"]

tokenized_train = tokenized_train.remove_columns(
    [c for c in tokenized_train.column_names if c not in keep_cols]
)

tokenized_valid = tokenized_valid.remove_columns(
    [c for c in tokenized_valid.column_names if c not in keep_cols]
)

tokenized_test = tokenized_test.remove_columns(
    [c for c in tokenized_test.column_names if c not in keep_cols]
)

In [24]:
print(tokenized_train[0])

{'input_ids': [162412, 265, 364, 187555, 259, 98923, 267, 55424, 347, 267, 336, 728, 259, 222797, 276, 259, 59315, 267, 926, 259, 262, 22163, 260, 83778, 265, 5516, 267, 259, 171880, 96913, 267, 59006, 5667, 267, 259, 38826, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [75336, 143474, 73589, 18358, 18422, 62394, 44029, 18422, 62394, 44029, 259, 70799, 

In [25]:
# A small sanity run

small_train = tokenized_train.select(range(200))
small_valid = tokenized_valid.select(range(50))

In [32]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_test_run",
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=2,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_valid,
    data_collator=data_collator
)

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan


TrainOutput(global_step=50, training_loss=0.0, metrics={'train_runtime': 12.1016, 'train_samples_per_second': 33.053, 'train_steps_per_second': 4.132, 'total_flos': 52874969088000.0, 'train_loss': 0.0, 'epoch': 2.0})

In [30]:
# Inspection bc training goes wrong]

print("Pad token id:", tokenizer.pad_token_id)
print("First 30 label ids:", tokenized_train[0]["labels"][:30])
print("Count of -100 in sample 0:", sum(1 for x in tokenized_train[0]["labels"] if x == -100))
print("Total label length:", len(tokenized_train[0]["labels"]))

Pad token id: 0
First 30 label ids: [75336, 143474, 73589, 18358, 18422, 62394, 44029, 18422, 62394, 44029, 259, 70799, 212675, 24216, 112829, 3610, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Count of -100 in sample 0: 47
Total label length: 64


In [31]:
sample_labels = tokenized_train[0]["labels"]
valid_tokens = [x for x in sample_labels if x != -100]
print("Non-masked label tokens:", valid_tokens[:20])
print("Number of non-masked label tokens:", len(valid_tokens))

Non-masked label tokens: [75336, 143474, 73589, 18358, 18422, 62394, 44029, 18422, 62394, 44029, 259, 70799, 212675, 24216, 112829, 3610, 1]
Number of non-masked label tokens: 17


In [34]:
# Manual forward pass to see what's wrong

import torch

model.train()

sample = small_train[0]

input_ids = torch.tensor([sample["input_ids"]]).to(model.device)
attention_mask = torch.tensor([sample["attention_mask"]]).to(model.device)
labels = torch.tensor([sample["labels"]]).to(model.device)

print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("labels shape:", labels.shape)
print("labels first 20:", labels[0][:20])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print("loss:", outputs.loss)
print("logits shape:", outputs.logits.shape)

input_ids shape: torch.Size([1, 128])
attention_mask shape: torch.Size([1, 128])
labels shape: torch.Size([1, 64])
labels first 20: tensor([ 75336, 143474,  73589,  18358,  18422,  62394,  44029,  18422,  62394,
         44029,    259,  70799, 212675,  24216, 112829,   3610,      1,   -100,
          -100,   -100], device='cuda:0')
loss: tensor(nan, device='cuda:0', grad_fn=<NllLossBackward0>)
logits shape: torch.Size([1, 64, 250112])


In [35]:
# Check if logits contain NaN

import torch

print("Any NaN in logits:", torch.isnan(outputs.logits).any().item())
print("Any Inf in logits:", torch.isinf(outputs.logits).any().item())
print("All logits finite:", torch.isfinite(outputs.logits).all().item())

Any NaN in logits: True
Any Inf in logits: False
All logits finite: False


In [36]:
print("Logits min:", torch.nan_to_num(outputs.logits).min().item())
print("Logits max:", torch.nan_to_num(outputs.logits).max().item())

Logits min: 0.0
Logits max: 0.0


In [37]:
# Testing the same sample on CPU bc the previous error is likely cause by the gpu

import torch

model_cpu = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model_cpu.eval()

sample = small_train[0]

input_ids_cpu = torch.tensor([sample["input_ids"]])
attention_mask_cpu = torch.tensor([sample["attention_mask"]])
labels_cpu = torch.tensor([sample["labels"]])

with torch.no_grad():
    outputs_cpu = model_cpu(
        input_ids=input_ids_cpu,
        attention_mask=attention_mask_cpu,
        labels=labels_cpu
    )

print("CPU loss:", outputs_cpu.loss)
print("CPU logits finite:", torch.isfinite(outputs_cpu.logits).all().item())

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


CPU loss: tensor(24.6329)
CPU logits finite: True


In [38]:
# Restart on a clean model

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# clear old state
del model
torch.cuda.empty_cache()

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to("cuda")

model.eval()

sample = small_train[0]

input_ids = torch.tensor([sample["input_ids"]], dtype=torch.long, device="cuda")
attention_mask = torch.tensor([sample["attention_mask"]], dtype=torch.long, device="cuda")
labels = torch.tensor([sample["labels"]], dtype=torch.long, device="cuda")

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("GPU loss:", outputs.loss)
print("Any NaN in logits:", torch.isnan(outputs.logits).any().item())
print("All logits finite:", torch.isfinite(outputs.logits).all().item())

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


GPU loss: tensor(24.6329, device='cuda:0')
Any NaN in logits: False
All logits finite: True


In [39]:
# New Trainer

from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_test_run_clean",
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=2,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_valid,
    data_collator=data_collator
)

In [40]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,28.878223,22.537510
2,26.316058,21.266447


TrainOutput(global_step=50, training_loss=27.854248657226563, metrics={'train_runtime': 18.2138, 'train_samples_per_second': 21.961, 'train_steps_per_second': 2.745, 'total_flos': 52874969088000.0, 'train_loss': 27.854248657226563, 'epoch': 2.0})